# Data Residency, Compliance, and Security in Deployment
Companion notebook for the **Data Residency, Compliance, and Security in Deployment** section of Chapter 12.

Move **less data**, not just data more securely -- a local summary plus a record ID, with details fetched on demand, beats shipping the full record to a central planner by default.

This notebook loads the real `service-account.yaml` shipped alongside it -- the book's least-privilege example, unchanged:

In [1]:
print(open("service-account.yaml").read())

serviceAccount:
  name: agent-reviewer
  permissions:
    - read:case-summary
    - write:audit-log
    - call:review-model
  deny:
    - read:raw-pii
    - call:billing-tool



The book shows the config but not the enforcement -- here's the permission check a gateway would actually run before letting a service act:

In [2]:
import yaml

account = yaml.safe_load(open("service-account.yaml"))["serviceAccount"]


def is_permitted(account: dict, action: str) -> bool:
    if action in account["deny"]:
        return False
    return action in account["permissions"]


for action in ["read:case-summary", "write:audit-log", "read:raw-pii", "call:billing-tool", "delete:case-summary"]:
    print(f"{is_permitted(account, action)!s:6s} <- {action}")

True   <- read:case-summary
True   <- write:audit-log
False  <- read:raw-pii
False  <- call:billing-tool
False  <- delete:case-summary


`read:raw-pii` and `call:billing-tool` are rejected even though nothing *requires* an explicit deny list to reject an unlisted action -- an unrecognized permission like `delete:case-summary` is already denied by not being in `permissions`. The `deny` list exists for the case that matters more: making sure a broader future permission grant can never accidentally include something this service account must never touch.

A small pre-flight check for the chapter's other rule -- review *what data crosses a new boundary* before adding a provider, tool, or deployment path:

In [3]:
def boundary_review(*, data_sent: list[str], destination: str, identity_verified: bool, logging_enabled: bool) -> dict:
    sensitive = {"raw-pii", "full-medical-record", "payment-credentials"}
    problems = []
    if sensitive & set(data_sent):
        problems.append(f"sensitive fields crossing boundary: {sensitive & set(data_sent)}")
    if not identity_verified:
        problems.append("destination identity/auth not verified")
    if not logging_enabled:
        problems.append("no audit logging on this path")
    return {"destination": destination, "approved": not problems, "problems": problems}


print(boundary_review(data_sent=["case_id", "summary"], destination="review-model",
                      identity_verified=True, logging_enabled=True))
print(boundary_review(data_sent=["raw-pii"], destination="third-party-analytics",
                      identity_verified=False, logging_enabled=False))

{'destination': 'review-model', 'approved': True, 'problems': []}
{'destination': 'third-party-analytics', 'approved': False, 'problems': ["sensitive fields crossing boundary: {'raw-pii'}", 'destination identity/auth not verified', 'no audit logging on this path']}
